# 02.8 — NaN handling

This dataset has 45,015 NaNs in a single trial — every removed channel is a NaN, by design. MATLAB let you barely notice; Python will not. This notebook is about the difference, and it closes Module 02 because it's where everything (arrays, tensors, autograd, losses) collides with this project's data reality.

You will learn:

* Why MATLAB habits make NaN feel safe — and why NumPy/PyTorch break those habits.
* The **poisoning** demo: one NaN in the loss kills every parameter in one step.
* Masking — the general pattern for computing around NaNs.
* This codebase's two-layer defense (Critical Note #38): zero at the input, mask in the loss — and why you need BOTH.

**Prerequisites:** [02.1](02.1_numpy_vs_matlab_arrays.ipynb), [02.5 autograd](02.5_autograd_basics.ipynb).

## Section 1 — What MATLAB does

MATLAB culture treats NaN as a friendly missing-data marker:

```matlab
mean(x, 'omitnan')       % explicit NaN-skipping is one flag away
nanmean(x)               % the older spelling
plot(x)                  % NaNs render as gaps — no error
max(x)                   % ignores NaN by default (!)
```

In this pipeline, NaN is *load-bearing*: `cgg_loadDataArray` keeps removed channels as NaN, and downstream code opts in to ignoring them where appropriate. Habits formed there transfer badly, because:

* **NumPy**: `np.mean` returns NaN if ANY input is NaN — skipping requires the separate `np.nanmean` family.
* **PyTorch**: same propagation, fewer `nan*` helpers — and autograd makes it worse, because NaN doesn't just propagate *forward*…

## Section 2 — The Python concepts you need

### 2.1 — Propagation: the default you must internalize

In [ ]:
import numpy as np
import torch

x = np.array([1.0, 2.0, np.nan, 4.0])
print("np.mean:    ", np.mean(x))        # nan — poisoned
print("np.nanmean: ", np.nanmean(x))     # 2.333 — the opt-in MATLAB-style behavior
print("np.max:     ", np.max(x))         # nan — even max! (MATLAB's max ignores NaN)

t = torch.tensor([1.0, 2.0, float("nan"), 4.0])
print("torch.mean:   ", t.mean().item())
print("torch.nanmean:", t.nanmean().item())

And the comparison trap that makes NaN *detection* non-obvious — NaN is not equal to anything, including itself:

In [ ]:
nan = float("nan")
print("nan == nan:", nan == nan)                          # False!
print("the right way:", torch.isnan(t).any().item())       # detection
print("count:", torch.isnan(t).sum().item(), "of", t.numel())

### 2.2 — The poisoning demo: one NaN kills the whole model

Forward propagation is the mild symptom. The fatal one involves autograd: a single NaN reaching the loss makes **every gradient in the graph NaN**, and after one `optimizer.step()` **every parameter is NaN — permanently**. Watch it happen:

In [ ]:
from torch import nn

torch.manual_seed(0)
model = nn.Linear(4, 1)
opt = torch.optim.SGD(model.parameters(), lr=0.1)

x = torch.randn(8, 4)
x[3, 2] = float("nan")            # ONE bad value, one sample, one feature
y = torch.randn(8, 1)

loss = nn.functional.mse_loss(model(x), y)
print("loss:", loss.item())        # already nan — forward propagation

loss.backward()
print("grad NaN fraction:", torch.isnan(model.weight.grad).float().mean().item())

opt.step()
print("weights after ONE step:", model.weight.data.flatten())

In [ ]:
# And there is no recovery — even on perfectly clean data, the model is dead:
clean = torch.randn(8, 4)
print("output on clean data:", model(clean).flatten()[:4])

This is why a NaN bug in training is so nasty: the crash site (loss goes NaN at some step) is far from the cause (one bad value in one batch), and everything after the poisoning is garbage. Diagnostics, in escalating order:

1. `torch.isnan(batch).any()` asserts at data-loading time — cheap, catches it at the source.
2. `torch.autograd.set_detect_anomaly(True)` — slow, but names the exact operation that produced the first NaN in backward.

### 2.3 — Masking: computing around NaN

The general pattern when NaN means "missing, exclude from computation" — build a boolean mask, compute only over valid entries:

In [ ]:
import matplotlib.pyplot as plt

# A synthetic (C=8, T=40) trial with two dead channels — miniature of the real data
g = torch.Generator().manual_seed(3)
trial = torch.randn(8, 40, generator=g)
trial[2, :] = float("nan")
trial[5, :] = float("nan")

mask = ~torch.isnan(trial)                       # True where data is VALID

fig, axes = plt.subplots(2, 1, figsize=(9, 4), sharex=True)
axes[0].imshow(torch.nan_to_num(trial), aspect="auto", cmap="viridis")
axes[0].set_title("trial data (dead channels 2 and 5 shown as 0 for display)")
axes[0].set_ylabel("channel")
axes[1].imshow(mask, aspect="auto", cmap="gray")
axes[1].set_title("validity mask — white = real data, black = NaN (excluded)")
axes[1].set_ylabel("channel"); axes[1].set_xlabel("time")
plt.tight_layout(); plt.show()

In [ ]:
# Masked statistics — the NaN-safe mean, by hand:
masked_mean = trial[mask].mean()
print("masked mean:  ", masked_mean.item())
print("nanmean agrees:", torch.nanmean(trial).item())

# Masked MSE between a prediction and a NaN-bearing target — the loss-side pattern:
pred = torch.zeros_like(trial)
mse_valid_only = ((pred - trial)[mask] ** 2).mean()
print("masked MSE:   ", mse_valid_only.item())

Three tools, three jobs:

| Tool | Job | Caution |
|---|---|---|
| `mask = ~torch.isnan(t)` + indexing | exclude NaN from a computation | the honest default |
| `torch.nan_to_num(t)` | replace NaN with 0 (or chosen values) | the zeros become *real data* to everything downstream — only safe if something else accounts for them |
| `torch.where(mask, t, fallback)` | conditional replacement | same caution as `nan_to_num` |

That caution row is the heart of the next section.

### 2.4 — The two-layer defense (Critical Note #38)

The production pipeline needs contradictory things from NaN:

* The **encoder** can't eat NaN (one forward pass would poison everything — §2.2). → **Layer 1: `NaNToZero`** at the composite's front door replaces NaN with 0. Zeros are harmless numerically, and the network learns dead channels carry no signal.
* The **reconstruction loss** must not treat those zeros as targets — otherwise the decoder gets punished unless it predicts 0 at every dead channel, wasting capacity on reproducing an artifact. → **Layer 2: masked MSE** recomputes the mask from the *original* input and excludes NaN positions from the loss entirely.

Either layer alone is wrong: zeroing without masking corrupts the training objective (silently — loss looks fine, model wastes capacity); masking without zeroing crashes the forward pass. Both together: the model sees clean numbers, the objective sees only real data. This is also why `MatFileTrialDataset` **preserves** NaN instead of cleaning it at load time — the loss needs the original NaN positions to build its mask.

## Section 3 — The `neural_data_decoding` implementation

Layer 1 — the input gate:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/models/layers/nan_to_zero.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if line.startswith("class NaNToZero"))
for k in range(i, min(i + 14, len(src))):
    print(f"{k + 1:4} | {src[k]}")

Layer 2 — the masked loss:

In [ ]:
src = Path("../../src/neural_data_decoding/training/losses/elbo.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if line.startswith("def masked_mse_reconstruction_loss"))
for k in range(i, min(i + 22, len(src))):
    print(f"{k + 1:4} | {src[k]}")

Things to spot:

* `NaNToZero` is an `nn.Module` with no parameters — pure transformation, but living in the module tree means it's part of the saved architecture, applied identically at train and eval.
* The masked loss takes the ORIGINAL (NaN-bearing) target and derives the mask from it — the division of labor from §2.4 in code.
* Read the docstrings: both cite Critical Note #38 and the MATLAB source they mirror. The normalization choice inside the loss (divide by batch size, not by `mask.sum()`) was verified **empirically against MATLAB** — the plan's original note had it wrong, and the parity probe caught it. Empiricism beats documentation.

## Section 4 — Hands-on exercises

### Exercise 4.1 — the wrong way, quantified

Using §2.3's `trial` (with its 2 dead channels of 8): compute the mean the WRONG way — `nan_to_num` first, then plain `mean()` — and compare to the masked mean. How large is the bias, and why is it exactly that size?

In [ ]:
# Your attempt here


In [ ]:
# Reveal:
wrong = torch.nan_to_num(trial).mean()
right = trial[~torch.isnan(trial)].mean()
print(f"wrong (zeros counted): {wrong.item():.4f}")
print(f"right (masked):        {right.item():.4f}")
print(f"ratio: {wrong.item() / right.item():.3f}  — exactly 6/8, because 2 of 8")
print("channels contribute zeros that dilute the sum while inflating the count.")

### Exercise 4.2 — guard a training loop

Write the one-line assertion you'd add at the top of a training step to catch NaN input *before* it poisons anything. Then make it fire.

In [ ]:
# Your attempt here


In [ ]:
# Reveal:
batch = torch.randn(4, 8)
batch[1, 3] = float("nan")
try:
    assert not torch.isnan(batch).any(), "NaN in input batch — check the data pipeline"
except AssertionError as e:
    print(f"caught before any damage: {e}")

## Section 5 — Common errors

### Loss is `nan` from step 0

NaN in the input data reaching the model raw. Assert at load time (Exercise 4.2); if the data legitimately contains NaN, you need the §2.4 treatment, not a cleanup.

### Loss becomes `nan` mid-training

Not usually data — usually numerics: exploding gradients (clip them; [02.7 §5](02.7_optimizers_and_learning_rates.ipynb)), a `log(0)` or `sqrt(0)` in a custom loss, or division by a near-zero denominator. `torch.autograd.set_detect_anomaly(True)` names the guilty operation.

### Model outputs NaN on all inputs, forever

You're post-poisoning (§2.2's second cell) — the parameters themselves are NaN. No amount of clean data recovers it; reload the last good checkpoint and fix the cause first.

### `nan == nan` comparisons silently never match

NaN is unequal to everything including itself (§2.1). Any filtering logic written with `==` lets NaN straight through — use `isnan`.

### Statistics look plausible but wrong

The Exercise 4.1 bias: `nan_to_num` + plain mean counts artificial zeros as observations. Plausible-but-diluted numbers are worse than a crash — nothing tells you. Masked stats or the `nan*` family, always.

### MATLAB parity fails only on NaN-bearing trials

The two pipelines disagree on *where* NaN is handled. Check the order: this project's convention is preserve-at-load, zero-at-encoder-input, mask-in-loss — Critical Note #38's exact division of labor.

## Section 6 — Further reading

- [IEEE 754 NaN semantics (Python docs on floating point)](https://docs.python.org/3/tutorial/floatingpoint.html) — why `nan != nan` is a standard, not a Python quirk.
- [torch.nanmean / nansum / nan_to_num](https://pytorch.org/docs/stable/generated/torch.nanmean.html) — PyTorch's (small) native NaN toolkit.
- [Anomaly detection docs](https://pytorch.org/docs/stable/autograd.html#anomaly-detection) — the NaN-in-backward debugger.
- [`training/losses/elbo.py`](../../src/neural_data_decoding/training/losses/elbo.py) — the production masked loss with its parity-verified normalization.

**Module 02 is complete.** Modules 03+ (data pipeline, architecture, training loop, loss orchestration…) continue in future batches — the [curriculum map](../README.md) tracks what's available. You now have every tool those modules assume.